# Claude Code as a headless worker

This notebook runs [Claude Code](https://code.claude.com/docs/en/overview) non-interactively in a prokube sandbox on a Git repository and shows its answer and the diff. The Anthropic API key never enters the sandbox. See the [README](./README.md) for how that works and for using Claude Code interactively in the sandbox.

Sandboxes must be enabled on your prokube cluster, the Lab needs sandbox access, and you need an Anthropic API key.

## Setup

In [ ]:
%pip install -q "prokube-sdk @ git+https://github.com/prokube/prokube-sdk@sandboxv3/main"

In a Lab with sandbox access, `SandboxClient()` needs no configuration: it uses the Lab's workspace and identity.

In [ ]:
from prokube.sandbox import SandboxClient

sandboxes = SandboxClient()
SANDBOX = "claude-code"

## Create the credentials

The key goes into the `anthropic-api-key` Secret. The next cell creates it unless it exists, and prompts for the key unless `ANTHROPIC_API_KEY` is set. The label `prokube.ai/sandbox-credential=true` makes the Secret available to the workspace's sandboxes at egress.

In [ ]:
import os
from getpass import getpass

secret = !kubectl get secret anthropic-api-key -o name --ignore-not-found
if not secret:
    anthropic_api_key = os.environ.get("ANTHROPIC_API_KEY") or getpass("Anthropic API key: ")
    !kubectl create secret generic anthropic-api-key --from-literal=ANTHROPIC_API_KEY={anthropic_api_key}
!kubectl label secret anthropic-api-key prokube.ai/sandbox-credential=true --overwrite

## Create the sandbox

Replace the preset's network rules: the Anthropic API, GitHub and npm, nothing else. Claude Code sends a placeholder key, and the egress gateway replaces the `x-api-key` header with the key from the Secret.

In [ ]:
sandboxes.create(SANDBOX)
sandboxes.set_network(SANDBOX, [
    {"https": {
        "hostnames": ["api.anthropic.com"],
        "ports": {"numbers": [443]},
        "effects": {"replaceHeaders": [{
            "header": "x-api-key",
            "credentialUri": sandboxes.secret_uri("anthropic-api-key", "ANTHROPIC_API_KEY"),
        }]},
    }},
    {"tlsPassthrough": {"hostnames": ["github.com", "registry.npmjs.org"], "ports": {"numbers": [443]}}},
])

Install Claude Code into `/workspace` and clone a small repository:

In [ ]:
result = sandboxes.run(SANDBOX, "npm install -q --prefix /workspace/claude-code @anthropic-ai/claude-code", timeout=600)
print(result.stdout, result.stderr)
result = sandboxes.run(SANDBOX, "git clone -q --depth 1 https://github.com/pypa/sampleproject /workspace/sampleproject")
print(result.stdout, result.stderr)

## Run Claude Code

`claude -p` runs one task and exits. `--permission-mode acceptEdits` lets it edit files but not run commands:

In [ ]:
import json
import shlex

TASK = "Add a function add_two to src/sample/simple.py, like add_one, and a test for it."

result = sandboxes.run(
    SANDBOX,
    f"/workspace/claude-code/node_modules/.bin/claude -p {shlex.quote(TASK)} "
    "--output-format json --permission-mode acceptEdits",
    cwd="/workspace/sampleproject",
    env={"ANTHROPIC_API_KEY": "placeholder"},
    timeout=600,
)
answer = json.loads(result.stdout)
print(answer["result"])
print(f"{answer['num_turns']} turns, ${answer['total_cost_usd']:.3f}")

The changes:

In [ ]:
print(sandboxes.run(SANDBOX, "git add -A && git diff --staged", cwd="/workspace/sampleproject").stdout)

## Clean up

Delete the sandbox. Keep the Secret if other examples use it; delete it with `kubectl delete secret anthropic-api-key` otherwise.

In [ ]:
sandboxes.delete(SANDBOX)
sandboxes.close()